In [1]:
import numpy as np
import pandas as pd

In [2]:
!pip install gensim


[notice] A new release of pip available: 22.3.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import gensim
import os

In [4]:
import pandas as pd

df = pd.read_csv("mtsamples.csv")

# Combine the description and transcription into a single massive text block for each row
# We use fillna('') to ensure we don't get errors if a row is missing a description
df['combined_text'] = df['description'].fillna('') + " " + df['transcription'].fillna('')

# Now, extract this combined column as your raw document list with removing missing values
raw_documents = df['combined_text'].dropna().tolist()

# Text Preprocessing

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

class ClinicalTextPreprocessor:
    def __init__(self):
        # Load standard English stop words
        self.stop_words = set(stopwords.words('english'))
        # Optional: Keep critical clinical negation words if needed
        # self.stop_words.remove('no')
        # self.stop_words.remove('not')

        self.lemmatizer = WordNetLemmatizer()

    def preprocess(self, text: str) -> list[str]:
        if not isinstance(text, str):
            return []

        # 1. Lowercase text
        text = text.lower()

        # 2. Remove punctuation and numbers (keep only alphabetic characters)
        text = re.sub(r'[^a-z\s]', ' ', text)

        # 3. Tokenize into words
        tokens = word_tokenize(text)

        # 4. Remove stop words, short noise (<2 letters), and apply Lemmatization
        cleaned_tokens = [
            self.lemmatizer.lemmatize(token)
            for token in tokens
            if token not in self.stop_words and len(token) > 2
        ]

        return cleaned_tokens

# --- Example Usage ---
preprocessor = ClinicalTextPreprocessor()

sample_note = "Patient presents with acute fractures and severe dyspnea. Prescribed 50mg Amoxicillin."
processed_tokens = preprocessor.preprocess(sample_note)



[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sunil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sunil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sunil\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sunil\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
processed_tokens

['patient',
 'present',
 'acute',
 'fracture',
 'severe',
 'dyspnea',
 'prescribed',
 'amoxicillin']

In [7]:
preprocessor = ClinicalTextPreprocessor()

tokenized_corpus = [preprocessor.preprocess(doc) for doc in raw_documents]

In [8]:
# Medical Phrases Handling ---- It turns ['myocardial', 'infarction'] into a single token ['myocardial_infarction']. It turns ['blood', 'pressure'] into ['blood_pressure']

from gensim.models.phrases import Phrases, Phraser

# Train the phrase detector on your corpus
# min_count: ignore phrases that happen very rarely
phrase_model = Phrases(tokenized_corpus, min_count=3, threshold=10)

# Freeze the model to make it faster (Phraser)
bigram_detector = Phraser(phrase_model)

# Apply the detector to glue our words together
phrased_corpus = [bigram_detector[doc] for doc in tokenized_corpus]

In [9]:
# Check total document count
print(f"Total Documents: {len(phrased_corpus)}")

# Inspect ONLY the first document, and ONLY its first 15 words
print("Sample of Document 1:", phrased_corpus[0][:15])

# Inspect total word count across the whole corpus safely
total_words = sum(len(doc) for doc in phrased_corpus)
print(f"Total Tokens in Corpus: {total_words:,}")

Total Documents: 4999
Sample of Document 1: ['year_old', 'white_female', 'present', 'complaint', 'allergy', 'subjective_year', 'old_white', 'female', 'present', 'complaint', 'allergy', 'used', 'allergy', 'lived', 'seattle']
Total Tokens in Corpus: 1,071,265


In [10]:
import pickle

# Save your phrased corpus to a compressed pickle file
with open("phrased_corpus.pkl", "wb") as f:
    pickle.dump(phrased_corpus, f)

print("Saved phrased_corpus.pkl successfully!")

Saved phrased_corpus.pkl successfully!


In [11]:
# import pickle

# # Load it back whenever you re-open Colab
# with open("phrased_corpus.pkl", "rb") as f:
#     phrased_corpus = pickle.load(f)

# print(f"Loaded {len(phrased_corpus)} documents ready for Word2Vec!")

In [12]:
from gensim.models import Word2Vec

# Train the model
# vector_size: how many dimensions represent each word (usually 100-300)
# window: how many words to look at before and after the target word
from gensim.models import Word2Vec

# Adjusted parameters for a smaller clinical dataset
model = Word2Vec(
    sentences= phrased_corpus, 
    vector_size=100, 
    window=5, 
    min_count=10,   # IMPORTANT: Ignores any word/phrase that appears fewer than 10 times
    epochs=50,      # IMPORTANT: Forces the model to read the 5,000 rows 50 times to learn better
    workers=4
)

# Test 
print(model.wv.most_similar('myocardial_infarction'))

[('segment_elevation', 0.45730116963386536), ('cad', 0.45527544617652893), ('heart_failure', 0.449823260307312), ('childhood', 0.44478851556777954), ('cerebrovascular_accident', 0.439162015914917), ('alcohol_abuse', 0.43232688307762146), ('cardiac_enzyme', 0.42424866557121277), ('failure', 0.42342451214790344), ('syndrome', 0.4130525290966034), ('artery_disease', 0.40869882702827454)]


In [13]:
print(model.wv.most_similar('pneumonia'))

[('treated_antibiotic', 0.5122978687286377), ('inclusive', 0.4899080693721771), ('pulmonary_medicine', 0.4869999587535858), ('renal_insufficiency', 0.4705018401145935), ('otitis_medium', 0.46472400426864624), ('bronchitis', 0.46288928389549255), ('likely_secondary', 0.45578500628471375), ('copd_exacerbation', 0.4551568627357483), ('hospital_acquired', 0.4539712369441986), ('pancreatitis', 0.447067528963089)]


In [14]:
model.wv.doesnt_match(['antibiotic','pneumonia','myocardial_infarction'])

'myocardial_infarction'

In [15]:
model.wv['allergy']

array([-1.93345380e+00, -2.95703745e+00, -5.77683806e-01,  2.66592288e+00,
        2.33290815e+00, -1.18342066e+00, -1.96426868e+00, -3.09031289e-02,
       -8.16529691e-01,  8.11466277e-01, -2.94927669e+00,  3.21014094e+00,
        2.72317410e+00,  2.00065947e+00,  4.60480452e+00, -3.35236311e+00,
       -1.29744017e+00,  1.22807220e-01, -1.12015200e+00, -1.70575380e-01,
        1.31416154e+00, -5.09361267e-01,  3.47823477e+00,  1.20814478e+00,
        9.18206349e-02, -3.68901938e-01, -7.71271646e-01, -3.66781592e-01,
       -3.73788506e-01,  7.03083724e-02,  3.82418364e-01,  1.62914491e+00,
        1.97213972e+00,  8.49745810e-01, -1.43110454e+00, -6.89402401e-01,
        3.31397486e+00, -2.99074388e+00, -6.24575949e+00, -2.78172076e-01,
        6.14743114e-01,  2.99646759e+00, -1.02240121e+00, -1.64747620e+00,
       -3.53401351e+00,  2.89697933e+00, -5.07488191e-01, -7.09205568e-02,
       -2.08512878e+00, -5.17057121e-01,  8.21783066e-01,  9.74024951e-01,
       -1.26897693e+00, -

In [16]:
model.wv.similarity('allergy','pneumonia')

np.float32(0.23367675)

In [17]:
model.wv.similarity('allergy','cough')

np.float32(0.26843712)

In [18]:
model.wv.get_normed_vectors()

array([[ 0.10247207, -0.08967727,  0.05855199, ...,  0.03289881,
        -0.00153012,  0.1204912 ],
       [-0.05146206,  0.1211629 ,  0.0082097 , ..., -0.00621216,
        -0.14485703, -0.14163417],
       [-0.0488121 ,  0.04092276, -0.02071336, ...,  0.00047676,
        -0.14193214, -0.13986814],
       ...,
       [-0.01688291,  0.12256708,  0.03483071, ..., -0.22873525,
        -0.09585435, -0.16074963],
       [-0.05671262, -0.06547151,  0.16388594, ..., -0.01382743,
         0.13620748, -0.02988176],
       [ 0.06717492, -0.15355334,  0.08110194, ...,  0.01232117,
         0.00205785,  0.0462553 ]], shape=(12371, 100), dtype=float32)

In [19]:
from gensim.models import Word2Vec

# 2. Query target medical terms
test_terms = ["pneumonia", "fracture", "hypertension", "amoxicillin"]

print("CLINICAL VECTOR EVALUATION")
for term in test_terms:
    if term in model.wv:
        print(f"\nTop 5 terms similar to '{term}':")
        similar = model.wv.most_similar(term, topn=5)
        for word, score in similar:
            print(f"  -> {word} (Similarity: {score:.4f})")
    else:
        print(f"\nTerm '{term}' not found in vocabulary.")

# 3. Test mathematical relationship (Cosine Similarity)
# Example: Check similarity between two related vs unrelated terms
sim_related = model.wv.similarity('pneumonia', 'cough')
sim_unrelated = model.wv.similarity('pneumonia', 'fracture')

print(f"\nSimilarity (pneumonia vs cough): {sim_related:.4f}")
print(f"\nSimilarity (pneumonia vs fracture): {sim_unrelated:.4f}")

CLINICAL VECTOR EVALUATION

Top 5 terms similar to 'pneumonia':
  -> treated_antibiotic (Similarity: 0.5123)
  -> inclusive (Similarity: 0.4899)
  -> pulmonary_medicine (Similarity: 0.4870)
  -> renal_insufficiency (Similarity: 0.4705)
  -> otitis_medium (Similarity: 0.4647)

Top 5 terms similar to 'fracture':
  -> internal_fixation (Similarity: 0.5292)
  -> fracture_dislocation (Similarity: 0.5173)
  -> bone_forearm (Similarity: 0.5157)
  -> comminuted (Similarity: 0.5118)
  -> fragment (Similarity: 0.5081)

Top 5 terms similar to 'hypertension':
  -> diabetes (Similarity: 0.6134)
  -> hypothyroidism (Similarity: 0.6050)
  -> hypercholesterolemia (Similarity: 0.5966)
  -> hypertension_hypercholesterolemia (Similarity: 0.5941)
  -> hypertension_hyperlipidemia (Similarity: 0.5553)

Top 5 terms similar to 'amoxicillin':
  -> augmentin (Similarity: 0.5210)
  -> avelox (Similarity: 0.5183)
  -> doxycycline (Similarity: 0.5114)
  -> cipro (Similarity: 0.5112)
  -> xopenex (Similarity: 0.501

In [20]:
y = model.wv.index_to_key

In [21]:
len(y) # unique vocabulary

12371

In [22]:
from sklearn.decomposition import PCA

In [23]:
pca = PCA(n_components=3)

In [24]:
X = pca.fit_transform(model.wv.get_normed_vectors())

In [25]:
X.shape

(12371, 3)

In [26]:
import plotly.express as px
fig = px.scatter_3d(X[100:200],x=0,y=1,z=2, color=y[100:200])
fig.show()

In [28]:
# Evaluation



import pandas as pd
from gensim.models import Word2Vec

# 1. Load your data and model
df = pd.read_csv("mtsamples.csv")


target_word = "pneumonia"
print(f"=== Evaluating context for: '{target_word}' ===")

# 2. Get the target word's actual medical specialties from the CSV
# Drop empty values and convert to string to avoid errors
df['transcription'] = df['transcription'].fillna("").astype(str)
target_matches = df[df['transcription'].str.contains(target_word, case=False)]
target_specialties = target_matches['medical_specialty'].value_counts().head(3)

print("\nTop 3 real-world specialties for this word:")
print(target_specialties)

# 3. Check if the model's predicted similar words align with those specialties
print("\nModel's Top 3 Similar Words:")
similar_words = model.wv.most_similar(target_word, topn=3)

for word, score in similar_words:
    word_matches = df[df['transcription'].str.contains(word, case=False)]
    word_specialties = word_matches['medical_specialty'].value_counts().head(1)
    
    top_spec = word_specialties.index[0] if not word_specialties.empty else "None"
    print(f" -> {word} (Similarity: {score:.2f}) | Most common specialty: {top_spec}")

=== Evaluating context for: 'pneumonia' ===

Top 3 real-world specialties for this word:
medical_specialty
Consult - History and Phy.    52
Cardiovascular / Pulmonary    41
General Medicine              27
Name: count, dtype: int64

Model's Top 3 Similar Words:
 -> common (Similarity: 0.87) | Most common specialty:  Surgery
 -> stapler_across (Similarity: 0.85) | Most common specialty: None
 -> biopsy_snare (Similarity: 0.82) | Most common specialty: None


In [2]:
# Save full model (includes network weights for continued training)
model.save("clinical_word2vec.model")

# Save ONLY the word vectors (smaller file size, faster loading for inference)
model.wv.save_word2vec_format("clinical_word2vec.kv", binary=True)
print("Model vectors successfully serialized for production deployment!")

Model vectors successfully serialized for production deployment!
